In [41]:
import glob
import pandas as pd
from pathlib import Path
from clearit.config import EMBEDDINGS_DIR, OUTPUTS_DIR


from pathlib import Path
import numpy as np
import pandas as pd

# Graph + Leiden + kNN/PCA (no scanpy needed)
import igraph as ig
import leidenalg as la
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors

# Reproducibility
SEED = 42
rng = np.random.default_rng(SEED)

# ======= Adjust these paths for your environment =======
BASE_DIR = Path.cwd()                         # change if you prefer a different project root
OUT_DIR  = OUTPUTS_DIR / "leiden"

# HDF5 files (concatenated tables prepared for MAPS benchmark)
H5_TNBC1 = EMBEDDINGS_DIR / "TNBC1-MxIF8" / "inForm_MC7" / "01_features-expressions" / "tnbc1-mxif8.hdf5"
H5_TNBC2 = EMBEDDINGS_DIR / "TNBC2-MIBI44" / "DeepCell_MC17" / "01_features-expressions" / "tnbc2-mibi8.hdf5"

# HDF5 keys (tables) — update if your store uses different names
H5_KEY_FEATURES   = "features"    # CLEAR-IT features per cell
H5_KEY_EXPR_SIZE  = "expressions"   # marker expression + cell size

# Column names (update to your schema if different)
ID_COL        = "cell_id"
PATIENT_COL   = "patient_id"
IMAGE_COL     = "image_id"        # or "tiff_id" / similar
PATCH_COL     = "patch_path"      # optional, if per-cell patches already exist
X_COL, Y_COL  = "x", "y"          # optional (pixel coords)
QUALITY_COL   = "quality_rank"    # optional (for QC filtering)

# Clustering defaults (we’ll revisit later)
PCA_DIMS      = 64
KNN_K         = 30
LEIDEN_RES    = 1.0

print("Cell 1 ready. Edit H5_TNBC1/H5_TNBC2 as needed before loading.")


Cell 1 ready. Edit H5_TNBC1/H5_TNBC2 as needed before loading.


In [6]:
# CELL 2A — Inspect HDF5 structure (both TNBC1 & TNBC2)

import pandas as pd
import h5py

def inspect_hdf5(h5_path):
    print(f"\n=== Inspecting: {h5_path} ===")
    # 1) Try pandas HDFStore keys
    try:
        with pd.HDFStore(h5_path, mode="r") as store:
            keys = store.keys()
            if keys:
                print("Pandas HDFStore keys:")
                for k in keys:
                    try:
                        st = store.get_storer(k)
                        nrows = getattr(st, "nrows", "?")
                        print(f"  {k}  (nrows≈{nrows})  -> pandas-format")
                    except Exception:
                        print(f"  {k}  -> (pandas-format, details unavailable)")
            else:
                print("Pandas HDFStore keys: (none)")
    except Exception as e:
        print(f"Pandas HDFStore open failed (not necessarily a problem): {e}")

    # 2) Generic HDF5 tree (h5py)
    print("h5py groups/datasets:")
    try:
        with h5py.File(h5_path, "r") as f:
            def walk(name, obj):
                if isinstance(obj, h5py.Group):
                    print(f"[GROUP]   /{name}")
                elif isinstance(obj, h5py.Dataset):
                    print(f"[DATASET] /{name:40s} shape={obj.shape} dtype={obj.dtype}")
            f.visititems(walk)
    except Exception as e:
        print(f"h5py open failed: {e}")

# Run on both paths so you can see what’s inside
inspect_hdf5(H5_TNBC1)
inspect_hdf5(H5_TNBC2)
print("\nTip: use the exact key/dataset printed above (including leading '/').")



=== Inspecting: /workspace/files/CLEAR-IT/embeddings/TNBC1-MxIF8/inForm_MC7/01_features-expressions/tnbc1-mxif8.hdf5 ===
Pandas HDFStore keys: (none)
h5py groups/datasets:
[GROUP]   /P01
[DATASET] /P01/expressions                          shape=(27193, 9) dtype=float32
[DATASET] /P01/features                             shape=(27193, 256) dtype=float32
[DATASET] /P01/labels                               shape=(27193,) dtype=int64
[GROUP]   /P02
[DATASET] /P02/expressions                          shape=(41954, 9) dtype=float32
[DATASET] /P02/features                             shape=(41954, 256) dtype=float32
[DATASET] /P02/labels                               shape=(41954,) dtype=int64
[GROUP]   /P03
[DATASET] /P03/expressions                          shape=(51054, 9) dtype=float32
[DATASET] /P03/features                             shape=(51054, 256) dtype=float32
[DATASET] /P03/labels                               shape=(51054,) dtype=int64
[GROUP]   /P04
[DATASET] /P04/expressions

In [30]:
# CELL 3 — Discover patients and define which to load (Train vs Test)

import h5py
from pathlib import Path

# Choose dataset
DATASET = "TNBC1"   # "TNBC1" or "TNBC2"
h5_path = H5_TNBC1 if DATASET.upper() == "TNBC1" else H5_TNBC2
assert Path(h5_path).exists(), f"Missing file: {h5_path}"

# List available patient groups (e.g., 'P01', 'P02', ...)
with h5py.File(h5_path, "r") as f:
    AVAILABLE_PTS = sorted([k for k in f.keys() if k.startswith("P")])
print(f"{DATASET}: {len(AVAILABLE_PTS)} patient groups found:\n{AVAILABLE_PTS}")

# ---- EDIT THIS SPLIT to your canonical Train/Test (or import from a file) ----
# Examples below are placeholders. Replace with your actual split.
TRAIN_PTS = [f"P{x:02d}" for x in range(1,48)] 
TEST_PTS = [f"P{x:02d}" for x in range(48,63)] 
# ------------------------------------------------------------------------------

# Sanity checks
unknown_train = sorted(set(TRAIN_PTS) - set(AVAILABLE_PTS))
unknown_test  = sorted(set(TEST_PTS) - set(AVAILABLE_PTS))
assert not unknown_train and not unknown_test, f"Unknown patients in split. Train: {unknown_train}, Test: {unknown_test}"

# Choose which set to load now (we’ll cluster on Train only)
SPLIT_TO_LOAD = "train"   # "train" or "test"
PATIENTS_TO_LOAD = TRAIN_PTS if SPLIT_TO_LOAD == "train" else TEST_PTS
print(f"Will load {len(PATIENTS_TO_LOAD)} patients from {SPLIT_TO_LOAD.upper()}: {PATIENTS_TO_LOAD if PATIENTS_TO_LOAD else '(none yet — fill TRAIN_PTS/TEST_PTS)'}")


TNBC1: 62 patient groups found:
['P01', 'P02', 'P03', 'P04', 'P05', 'P06', 'P07', 'P08', 'P09', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P20', 'P21', 'P22', 'P23', 'P24', 'P25', 'P26', 'P27', 'P28', 'P29', 'P30', 'P31', 'P32', 'P33', 'P34', 'P35', 'P36', 'P37', 'P38', 'P39', 'P40', 'P41', 'P42', 'P43', 'P44', 'P45', 'P46', 'P47', 'P48', 'P49', 'P50', 'P51', 'P52', 'P53', 'P54', 'P55', 'P56', 'P57', 'P58', 'P59', 'P60', 'P61', 'P62']
Will load 47 patients from TRAIN: ['P01', 'P02', 'P03', 'P04', 'P05', 'P06', 'P07', 'P08', 'P09', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P20', 'P21', 'P22', 'P23', 'P24', 'P25', 'P26', 'P27', 'P28', 'P29', 'P30', 'P31', 'P32', 'P33', 'P34', 'P35', 'P36', 'P37', 'P38', 'P39', 'P40', 'P41', 'P42', 'P43', 'P44', 'P45', 'P46', 'P47']


In [31]:
# CELL 4 — Load per-patient features (+ optional expressions/labels), with subsampling caps

import h5py
import numpy as np
import pandas as pd

# ---- Controls (edit as needed) ----
LOAD_EXPRESSIONS = False   # True to also load /Px/expressions
LOAD_LABELS      = False   # True to also load /Px/labels (ground-truth classes)
PER_PATIENT_CAP  = 10000   # max cells per patient (None for no cap)
GLOBAL_CAP       = 150000  # max total cells across patients (None for no cap)
RANDOM_STATE     = 42
# -----------------------------------

rng_local = np.random.default_rng(RANDOM_STATE)

def _cap_indices(n, per_patient_cap, global_remaining):
    """Return indices to keep given per-patient and global caps."""
    take = n
    if per_patient_cap is not None:
        take = min(take, per_patient_cap)
    if global_remaining is not None:
        take = min(take, global_remaining)
    if take >= n:
        return np.arange(n)
    # random subsample
    return rng_local.choice(n, size=take, replace=False)

records = []
X_blocks = []            # features
E_blocks = []            # expressions (optional)
y_blocks = []            # labels (optional)

global_remaining = None if GLOBAL_CAP is None else int(GLOBAL_CAP)

with h5py.File(h5_path, "r") as f:
    for pid in PATIENTS_TO_LOAD:
        grp = f[pid]
        # Required: features
        assert "features" in grp, f"Missing '{pid}/features'"
        feats = grp["features"][()]  # shape (Ni, D)
        n_i = feats.shape[0]

        # Optional: expressions + labels
        expr = grp["expressions"][()] if (LOAD_EXPRESSIONS and "expressions" in grp) else None
        labl = grp["labels"][()]      if (LOAD_LABELS and "labels" in grp)      else None

        # Decide how many to take for this patient
        keep_idx = _cap_indices(n_i, PER_PATIENT_CAP, global_remaining)
        if global_remaining is not None:
            global_remaining -= len(keep_idx)
            if global_remaining <= 0:
                # we still finalize current patient’s selection, then break
                pass

        # Slice arrays
        feats_keep = feats[keep_idx]
        X_blocks.append(feats_keep)

        if expr is not None:
            E_blocks.append(expr[keep_idx])
        if labl is not None:
            y_blocks.append(labl[keep_idx])

        # Metadata rows
        records.extend([{"patient": pid, "idx_within_patient": int(i)} for i in keep_idx])

        # Stop if we’ve hit the global cap
        if global_remaining is not None and global_remaining <= 0:
            break

# Concatenate
X = np.vstack(X_blocks) if X_blocks else np.empty((0, 0), dtype=np.float32)
meta = pd.DataFrame.from_records(records)
print(f"Loaded features: X shape = {X.shape}, meta rows = {len(meta)}")

if LOAD_EXPRESSIONS and E_blocks:
    E = np.vstack(E_blocks)
    print(f"Loaded expressions: shape = {E.shape}")
else:
    E = None

if LOAD_LABELS and y_blocks:
    y = np.concatenate(y_blocks)
    print(f"Loaded labels: shape = {y.shape}")
else:
    y = None

# Quick peek
display(meta.head())
print("Feature dimensionality:", X.shape[1] if X.size else "(empty)")


Loaded features: X shape = (150000, 256), meta rows = 150000


,patient,idx_within_patient
0,P01,10578
1,P01,23373
2,P01,26081
3,P01,17141
4,P01,8102


Feature dimensionality: 256


In [32]:
# CELL 5 — Standardize and PCA
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import numpy as np

assert X.ndim == 2 and X.shape[0] == len(meta), "X/meta mismatch."

PCA_DIMS = 64  # you can tweak (e.g., 64/96/128)

scaler = StandardScaler(with_mean=True, with_std=True)
Xz = scaler.fit_transform(X)  # (N, D)

pca = PCA(n_components=PCA_DIMS, random_state=SEED, svd_solver="auto")
X_pca = pca.fit_transform(Xz)  # (N, PCA_DIMS)

print(f"PCA: explained variance ratio (first 10) = {np.round(pca.explained_variance_ratio_[:10], 4)}")
print(f"PCA: cumulative (k={PCA_DIMS}) = {pca.explained_variance_ratio_[:PCA_DIMS].sum():.4f}")


PCA: explained variance ratio (first 10) = [0.15   0.1034 0.0696 0.0688 0.0535 0.0433 0.0374 0.0312 0.0285 0.0242]
PCA: cumulative (k=64) = 0.9575


In [33]:
# CELL 6 — Build kNN graph and run Leiden
from sklearn.neighbors import NearestNeighbors
import igraph as ig
import leidenalg as la
import numpy as np

KNN_K = 30        # typical 20–50
LEIDEN_RES = 1.0  # try {0.8, 1.0, 1.2, 1.5}

# kNN on PCA space
nn = NearestNeighbors(n_neighbors=KNN_K, metric="euclidean", n_jobs=-1)
nn.fit(X_pca)
dists, nbrs = nn.kneighbors(X_pca, return_distance=True)  # shapes: (N, K), (N, K)

# Build an undirected edge list (symmetrize kNN)
N = X_pca.shape[0]
rows = np.repeat(np.arange(N), KNN_K)
cols = nbrs.ravel()
weights = 1.0 / (1.0 + dists.ravel())  # simple similarity (positive)

# Keep i != j
mask = rows != cols
rows, cols, weights = rows[mask], cols[mask], weights[mask]

# Symmetrize: keep max weight for each undirected pair
# Represent edges as sorted tuple (min, max)
pairs = np.minimum(rows, cols), np.maximum(rows, cols)
edge_keys = np.stack(pairs, axis=1)
# aggregate by unique pairs
# (vectorized unique with argmax of weight per pair)
import pandas as pd
edges_df = pd.DataFrame({
    "u": edge_keys[:,0],
    "v": edge_keys[:,1],
    "w": weights
})
edges_df = edges_df.sort_values(["u","v","w"], ascending=[True, True, False]).drop_duplicates(["u","v"], keep="first")

g = ig.Graph(n=N, edges=list(zip(edges_df.u.values, edges_df.v.values)), directed=False)
g.es["weight"] = edges_df.w.values

# Leiden (RBConfigurationVertexPartition with resolution_parameter)
part = la.find_partition(
    g,
    la.RBConfigurationVertexPartition,
    weights=g.es["weight"],
    resolution_parameter=LEIDEN_RES,
    seed=SEED,
)
labels_leiden = np.array(part.membership, dtype=int)
n_clusters = int(labels_leiden.max() + 1)

meta = meta.copy()
meta["cluster_id"] = labels_leiden
print(f"Leiden done: {n_clusters} clusters")
print(meta["cluster_id"].value_counts().sort_index().head(20))


Leiden done: 22 clusters
cluster_id
0     13883
1     11914
2     11697
3     11120
4     10621
5      9804
6      9668
7      8244
8      7996
9      6910
10     6702
11     6464
12     6146
13     5687
14     4771
15     4521
16     4185
17     2629
18     2528
19     2324
Name: count, dtype: int64


In [34]:
# CELL 7 — Quick cluster summary
import pandas as pd

cluster_sizes = meta["cluster_id"].value_counts().sort_index()
summary = (
    meta.groupby("cluster_id")["patient"]
        .nunique()
        .rename("n_patients")
        .to_frame()
        .assign(size=cluster_sizes.values)
        .reset_index()
        .sort_values("size", ascending=False)
)

print(f"Total clusters: {summary.shape[0]}")
display(summary.head(20))


Total clusters: 22


,cluster_id,n_patients,size
0,0,16,13883
1,1,14,11914
2,2,16,11697
3,3,15,11120
4,4,13,10621
5,5,16,9804
6,6,14,9668
7,7,13,8244
8,8,9,7996
9,9,13,6910


In [35]:
# CELL 8 — Exemplar selection (10 per cluster, diversify by patient)
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors

EXEMPLARS_PER_CLUSTER = 10
EXEMPLARS_PER_PATIENT_MAX = 2
DENSITY_K = 15  # local density within cluster computed via kNN in PCA space

meta = meta.copy()

# Precompute per-cluster centroids in PCA space
centroids = (
    pd.DataFrame(X_pca)
    .assign(cluster_id=meta["cluster_id"].values)
    .groupby("cluster_id")
    .mean()
    .values
)  # shape: (K, PCA_DIMS)

# For each cluster, compute medoid-distance and local density
rows = []
for cid in sorted(meta["cluster_id"].unique()):
    idx = np.where(meta["cluster_id"].values == cid)[0]
    Xc = X_pca[idx]

    # distance to centroid (smaller is better)
    c = centroids[cid][None, :]
    dist_centroid = np.linalg.norm(Xc - c, axis=1)

    # local density: inverse mean distance to DENSITY_K nearest neighbors within the cluster
    k_loc = min(DENSITY_K, max(1, len(idx) - 1))
    nnc = NearestNeighbors(n_neighbors=k_loc+1, metric="euclidean").fit(Xc)
    dloc, _ = nnc.kneighbors(Xc)  # includes self at 0
    # exclude self (first column)
    dloc = dloc[:, 1:] if dloc.shape[1] > 1 else dloc
    local_density = 1.0 / (1e-8 + dloc.mean(axis=1))

    # rank combination: lower distance rank + higher density rank
    r1 = pd.Series(dist_centroid).rank(method="average", ascending=True).values
    r2 = pd.Series(local_density).rank(method="average", ascending=False).values
    combo = r1 + r2

    sub = pd.DataFrame({
        "global_idx": idx,
        "cluster_id": cid,
        "dist_centroid": dist_centroid,
        "local_density": local_density,
        "rank_score": combo
    })
    rows.append(sub)

ex_df = pd.concat(rows, ignore_index=True)

# Sort by rank within cluster
ex_df["rank_within_cluster"] = ex_df.groupby("cluster_id")["rank_score"].rank(method="first")
ex_df = ex_df.sort_values(["cluster_id", "rank_within_cluster"])

# Enforce per-patient diversification
def take_with_patient_caps(df_cluster, max_total, per_patient_max):
    taken = []
    per_pt_counts = {}
    for _, r in df_cluster.iterrows():
        gid = int(r["global_idx"])
        pid = meta.iloc[gid]["patient"]
        if per_pt_counts.get(pid, 0) >= per_patient_max:
            continue
        taken.append(gid)
        per_pt_counts[pid] = per_pt_counts.get(pid, 0) + 1
        if len(taken) >= max_total:
            break
    return taken

chosen_indices = []
for cid, sub in ex_df.groupby("cluster_id", sort=True):
    selected = take_with_patient_caps(sub, EXEMPLARS_PER_CLUSTER, EXEMPLARS_PER_PATIENT_MAX)
    chosen_indices.extend(selected)

exemplars = meta.iloc[chosen_indices].copy()
exemplars["exemplar_rank"] = (
    ex_df.set_index("global_idx").loc[chosen_indices, "rank_within_cluster"].values
)

print(f"Selected {len(exemplars)} exemplars across {exemplars['cluster_id'].nunique()} clusters "
      f"(~{EXEMPLARS_PER_CLUSTER} per cluster).")
display(exemplars.head(20))


Selected 220 exemplars across 22 clusters (~10 per cluster).


,patient,idx_within_patient,cluster_id,exemplar_rank
40139,P05,24246,0,1.0
46264,P05,22995,0,2.0
56073,P06,13731,0,6.0
58204,P06,26065,0,7.0
77257,P08,4366,0,32.0
103339,P11,3339,0,60.0
118530,P13,23511,0,83.0
119305,P13,21497,0,86.0
133100,P14,20236,0,87.0
70144,P08,3864,0,90.0


In [40]:
# CELL 9 — Save cluster assignments and exemplars as CSV
from pathlib import Path

OUT_DIR.mkdir(parents=True, exist_ok=True)

assign_csv    = OUT_DIR / f"{DATASET.lower()}_train_clusters.csv"
exemplars_csv = OUT_DIR / f"{DATASET.lower()}_train_exemplars.csv"

# Save the full metadata (includes patient + cluster_id, etc.)
meta.to_csv(assign_csv, index=False)

# Save exemplars separately (these are the 10-per-cluster for expert review)
exemplars.to_csv(exemplars_csv, index=False)

print("Saved:")
print(" -", assign_csv)
print(" -", exemplars_csv)


Saved:
 - /workspace/files/CLEAR-IT/outputs/leiden/tnbc1_train_clusters.csv
 - /workspace/files/CLEAR-IT/outputs/leiden/tnbc1_train_exemplars.csv
